In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, LogFormatterMathtext

file_path = "eeg_case_study_outputs_no_normalization/eeg_summary_by_subject_and_method.csv"
df = pd.read_csv(file_path)

method_map = {
    "Proposed algorithm": "FO-GS",
    "Chatterjee baseline": "FO-BS",
    "Wavelet+OLS baseline": "FO-WT",
}

df = df[df["method"].isin(method_map.keys())].copy()
df["method_short"] = df["method"].map(method_map)

df["subject"] = df["file_name"].str.replace(".csv", "", regex=False)
df["subject_idx"] = df["subject"].str.extract(r"(\d+)")[0].astype(int)
df = df.sort_values(["subject_idx", "method_short"]).reset_index(drop=True)

subjects = sorted(df["subject"].unique(), key=lambda x: int(x[1:]))

ylabel_map = {
    "train_nmse_mean": "training one-step NMSE",
    "test_nmse_mean": "test one-step NMSE",
}

plot_order = ["FO-GS", "FO-BS", "FO-WT"]

colors = {
    "FO-GS": "#FF0000",   # red
    "FO-BS": "#0000FF",   # blue
    "FO-WT": "#17BECF",   # cyan
}

x = np.arange(len(subjects))
bar_width = 0.25

pivot_df = df.pivot(index="subject", columns="method_short")
pivot_df = pivot_df.loc[subjects]


def plot_single_metric(metric, save_path):
    fig, ax = plt.subplots(figsize=(16, 5))

    handles = []
    labels = []

    for j, method in enumerate(plot_order):
        if method in pivot_df[metric].columns:
            offset = (j - 1) * bar_width
            bars = ax.bar(
                x + offset,
                pivot_df[metric][method].values,
                width=bar_width,
                color=colors[method],
                label=method,
                align="center",
            )
            handles.append(bars[0])
            labels.append(method)

    ax.set_xlabel("EEG subject ID", fontsize=16)
    ax.set_ylabel(ylabel_map[metric], fontsize=16)
    ax.set_yscale("log")

    ax.set_xticks(x)
    ax.set_xticklabels(subjects, rotation=45, ha="center")
    ax.set_xlim(-0.5, len(subjects) - 0.5)

    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", labelsize=16)

    # Manually control y-axis ticks
    if metric == "train_nmse_mean":
        ax.set_ylim(4e-4, 1e-2)
        ax.yaxis.set_major_locator(FixedLocator([1e-3, 1e-2]))
        ax.yaxis.set_major_formatter(LogFormatterMathtext())

    elif metric == "test_nmse_mean":
        ax.set_ylim(1, 1e4)
        ax.yaxis.set_major_locator(FixedLocator([1e1, 1e2, 1e3, 1e4]))
        ax.yaxis.set_major_formatter(LogFormatterMathtext())

    # Legend
    ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.18),
        ncol=3,
        fontsize=16,
        frameon=False,
    )

    plt.tight_layout()
    fig.savefig(save_path, bbox_inches="tight")
    plt.show()


# Generate training figure
plot_single_metric(
    metric="train_nmse_mean",
    save_path="eeg_train_nmse_comparison.pdf"
)

# Generate test figure
plot_single_metric(
    metric="test_nmse_mean",
    save_path="eeg_test_nmse_comparison.pdf"
)